In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class LESSArgs:
    data_dir: str
    model_path: str
    percentage: float = 1.0
    data_seed: int = 0
    job_name: str = "default_job"
    

    output_dir: Optional[str] = None  # will be set based on job_name if None
    
    # Train files - set in __post_init__
    train_files: List[str] = field(default_factory=list)
    

    do_train: bool = True
    max_seq_length: int = 2048
    use_fast_tokenizer: bool = True
    lr_scheduler_type: str = "linear"
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    evaluation_strategy: str = "no"
    logging_steps: int = 1
    save_strategy: str = "epoch"  # note was 'no' but final is 'epoch'
    num_train_epochs: int = 4
    bf16: bool = True
    tf32: bool = False
    fp16: bool = False
    overwrite_output_dir: bool = True
    report_to: str = "wandb"
    optim: str = "adamw_torch"
    seed: int = 0
    lora: bool = True
    lora_r: int = 128
    lora_alpha: int = 512
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    learning_rate: float = 2e-5
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 32
    
    # FSDP options - defaults to None unless model_path matches
    fsdp: Optional[str] = None
    fsdp_config: Optional[str] = None

    def __post_init__(self):
        if self.output_dir is None:
            self.output_dir = f"../out/{self.job_name}"
        
        # Set train files based on data_dir
        self.train_files = [
            f"{self.data_dir}/train/processed/flan_v2/flan_v2_data.jsonl",
            f"{self.data_dir}/train/processed/cot/cot_data.jsonl",
            f"{self.data_dir}/train/processed/dolly/dolly_data.jsonl",
            f"{self.data_dir}/train/processed/oasst1/oasst1_data.jsonl",
        ]
        
        # Setup FSDP config based on model_path
        if self.model_path == "meta-llama/Llama-2-13b-hf":
            self.fsdp = "full_shard auto_wrap"
            self.fsdp_config = "llama2_13b_finetune"
        elif self.model_path == "mistralai/Mistral-7B-v0.1":
            self.fsdp = "full_shard auto_wrap"
            self.fsdp_config = "mistral_7b_finetune"


In [ ]:
args = LESSArgs(
    data_dir="./less/data",
    model_path="meta-llama/Llama-2-7b-hf",
    percentage=0.8,
    data_seed=42,
    job_name="test_job"
)
args

LESSArgs(data_dir='/path/to/data', model_path='meta-llama/Llama-2-13b-hf', percentage=0.8, data_seed=42, job_name='test_job', output_dir='../out/test_job', train_files=['/path/to/data/train/processed/flan_v2/flan_v2_data.jsonl', '/path/to/data/train/processed/cot/cot_data.jsonl', '/path/to/data/train/processed/dolly/dolly_data.jsonl', '/path/to/data/train/processed/oasst1/oasst1_data.jsonl'], do_train=True, max_seq_length=2048, use_fast_tokenizer=True, lr_scheduler_type='linear', warmup_ratio=0.03, weight_decay=0.0, evaluation_strategy='no', logging_steps=1, save_strategy='epoch', num_train_epochs=4, bf16=True, tf32=False, fp16=False, overwrite_output_dir=True, report_to='wandb', optim='adamw_torch', seed=0, lora=True, lora_r=128, lora_alpha=512, lora_dropout=0.1, lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], learning_rate=2e-05, per_device_train_batch_size=1, gradient_accumulation_steps=32, fsdp='full_shard auto_wrap', fsdp_config='llama2_13b_finetune')